#### 이상치제거추가_데이터전처리_로지스틱분류실험 251103

- 251029의 실험결과가 너무 안좋아서 추가 데이터 전처리 필요할듯

- 이상치 제거 방법: IQR, One-Class SVM, Local Outlier Factor, Isolation Forest 등 여러가지가 있는듯 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("./credit_risk_dataset.csv") 
df.head() 

In [ ]:
df.shape

- loan_int_rate 변수 제거 

In [ ]:
corr_x = df.select_dtypes(['float', 'int']).corr()

In [ ]:
print("person_age와 cb_person_cred_hist_lengh 사이에 상관관계 있어보이므로 나중에 실험필요")
plt.figure(figsize=(6, 6))  
sns.heatmap(corr_x, annot=True, cmap='Reds')  
plt.show()

In [ ]:
df = df.drop(['loan_int_rate'], axis=1) 
df.head() 

#### 결측치 확인 및 채워넣기 

In [ ]:
df.isnull().sum()

In [ ]:
df = df.fillna(0)
df.isnull().sum()

In [ ]:
df.select_dtypes('object').head()

#### 이상치 box plot 시각화 

In [ ]:
df.columns

In [ ]:
df.select_dtypes('number').head()

In [ ]:
now_columns = list(df.select_dtypes('number').columns)
now_columns.remove('loan_status')
now_columns

In [ ]:
print("loan_status는 0,1인 변수이므로 이것을 제외하고 시각화.")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 6))
axes = axes.flatten()

for i, col in enumerate(now_columns):
    sns.boxplot(data=df, y = col, ax=axes[i], color='cornflowerblue')
    axes[i].set_title(f'{col}')

plt.tight_layout()
plt.show()

In [ ]:
print("위 6개 컬럼들은 다 이상치가 있어보이므로 제거 필요. ")

#### 값 파악 

In [ ]:
print(df['person_age'].max())
print(df[df['person_age'] >= 100].shape[0])
print(df[df['person_age'] >= 100].loan_grade.unique())  # 데이터 많은쪽 클래스라 제거해도 될듯 
print("B,C는 데이터 많은쪽 클래스라 100세 이상인 데이터 제거해도 될듯. ")

In [ ]:
print(df['person_income'].max())
print(df['person_income'].sort_values(ascending=False))
print(df[df['person_income'] == 6000000].loan_grade.unique())
print("C는 데이터 많은쪽 클래스라 데이터 제거해도 될듯. ")

#### 데이터 나누기 

In [ ]:
df.columns

In [ ]:
features = list(df.columns)
features.remove('loan_grade')

X = df[features]
y = df['loan_grade']

#### 명목형변수 변경   
- 우선은 one-hot-encoding으로 / label encoding도 고려해보기 

In [ ]:
X.select_dtypes('object').head()

In [ ]:
# one-hot encoding
# X = pd.get_dummies(X, dtype=float)
# X.shape

In [ ]:
# label encoding
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X['person_home_ownership'] = encoder.fit_transform(X['person_home_ownership'])
X['loan_intent'] = encoder.fit_transform(X['loan_intent'])
X['cb_person_default_on_file'] = encoder.fit_transform(X['cb_person_default_on_file'])

In [ ]:
X.info()

In [ ]:
# 트레이닝/테스트 데이터 분할 
from sklearn.model_selection import train_test_split
X_tn, X_te, y_tn, y_te = train_test_split(X, y, random_state=0, stratify=y)

In [ ]:
# 데이터 표준화
from sklearn.preprocessing import StandardScaler 
std_scale = StandardScaler()
std_scale.fit(X_tn)
X_tn_std = std_scale.transform(X_tn)
X_te_std = std_scale.transform(X_te)

In [ ]:
y_tn.shape

In [ ]:
df_tmp = pd.concat([pd.DataFrame(X_tn_std, columns=X.columns), pd.Series(y_tn)], axis=1)
df_tmp.head()

In [ ]:
df_tmp.shape

In [ ]:
x_cols = list(df.columns)
x_cols.remove('loan_grade')

In [ ]:
len(x_cols)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(12, 12))
axes = axes.flatten()

for i, col in enumerate(x_cols):
    sns.histplot(data=df_tmp, x=col, hue='loan_grade', bins=20, kde=True, ax=axes[i], color='cornflowerblue')
    axes[i].set_title(f'{col} Histogram')

plt.tight_layout()
plt.show()

#### 로지스틱 모델 돌려보기 

In [ ]:
# 로지스틱 회귀분석 (L2 제약식 적용)
from sklearn.linear_model import LogisticRegression
clf_logi_l2 = LogisticRegression(penalty='l2')
clf_logi_l2.fit(X_tn_std, y_tn)

In [ ]:
# 로지스틱 회귀분석 모형 (L2 제약식 적용) 추정 계수
print(clf_logi_l2.coef_)
print(clf_logi_l2.intercept_)

In [ ]:
# 예측
pred_logistic = clf_logi_l2.predict(X_te_std)
print(pred_logistic)

In [ ]:
# confusion matrix 확인
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_te, pred_logistic)
print(conf_matrix)

In [ ]:
labels = sorted(df['loan_grade'].unique())
print(labels)

plt.figure(figsize=(4,4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels, yticklabels=labels, 
            linewidths=0.5, cbar=False, square=True)

plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# 분류 레포트 확인
from sklearn.metrics import classification_report
class_report = classification_report(y_te, pred_logistic)
print(class_report)

In [ ]:
print("loan_int_rate변수 제거하니까 성능 엄청낮음.\n그리고 클래스별로 비율이 달라서인지 undersampling이 필요할 것 같음.")
print("클래스가 다양하니까 데이터도 더 많이 필요할것같고, one-hot encoding보다는 labelencoder가 필요할 것 같음.")
print("labelEncoder로 해도 성능 잘 안오름 ..")

In [ ]:
print("이상치를 조금 제거하고 돌려봐야하나 싶음")